# Modelo

Para ver como cambia la performance del modelo con el tratamiento de los datos primero hay que tener un modelo de partida que optimizar.

In [103]:
import librosa
import numpy as np
import pandas as pd
import os
from scipy.signal import butter, filtfilt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import recall_score, classification_report, confusion_matrix, roc_auc_score

from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold

from sklearn.decomposition import PCA

In [104]:
df = pd.read_csv('./icbhi/ICBHI_ciclos/ICBHI_ciclos_metadata.csv')

In [105]:
df.head()

,patient,recording_index,location,mode,instrument,cycle_number,start_sec,end_sec,duration_sec,crackles,wheezes,cycle_wav_file,wav_base_file,age,sex,adult_bmi,child_weight,child_height,diagnosis
0,101,1b1,Al,sc,Meditron,1,0.036,0.579,0.543,0,0,101_1b1_Al_sc_Meditron_cycle1.wav,101_1b1_Al_sc_Meditron.wav,3.0,F,NaN,19.0,99.0,URTI
1,101,1b1,Al,sc,Meditron,2,0.579,2.450,1.871,0,0,101_1b1_Al_sc_Meditron_cycle2.wav,101_1b1_Al_sc_Meditron.wav,3.0,F,NaN,19.0,99.0,URTI
2,101,1b1,Al,sc,Meditron,3,2.450,3.893,1.443,0,0,101_1b1_Al_sc_Meditron_cycle3.wav,101_1b1_Al_sc_Meditron.wav,3.0,F,NaN,19.0,99.0,URTI
3,101,1b1,Al,sc,Meditron,4,3.893,5.793,1.900,0,0,101_1b1_Al_sc_Meditron_cycle4.wav,101_1b1_Al_sc_Meditron.wav,3.0,F,NaN,19.0,99.0,URTI
4,101,1b1,Al,sc,Meditron,5,5.793,7.521,1.728,0,0,101_1b1_Al_sc_Meditron_cycle5.wav,101_1b1_Al_sc_Meditron.wav,3.0,F,NaN,19.0,99.0,URTI


Vamos a hacer un modelo binario normal / anormal

In [106]:
df['label'] = np.where((df['crackles'] == 0) & (df['wheezes'] == 0), 0, 1)

In [107]:
df.label.value_counts()

label
0    3642
1    3256
Name: count, dtype: int64

Queda bien balanceado, luego habría que ver temas de equidad.

Voy a guardar un df alternativo con solo los audios del AKGC... y Tc

In [108]:
df_filtrado = df[(df['instrument'] == 'AKGC417L') | (df['location'] == 'Tc')].copy()

In [109]:
df_filtrado.label.value_counts()

label
1    2465
0    2143
Name: count, dtype: int64

In [110]:
target = df['label'].values

## Espectrogramas

Se suelen usar Mel espectrogramas

In [ ]:
def lowpass_filter(y, sr, cutoff_hz=CUTOFF_HZ, order=6):
    nyq = 0.5 * sr
    normal_cutoff = cutoff_hz / nyq
    b, a = butter(order, normal_cutoff, btype='low', analog=False)
    y_filt = filtfilt(b, a, y)
    return y_filt


def filter_and_resample(path, sr_target=SAMPLING_RATE):
    y, sr = librosa.load(path, sr=None)
    y = lowpass_filter(y, sr, cutoff_hz=CUTOFF_HZ)
    if sr != sr_target:
        y = librosa.resample(y, orig_sr=sr, target_sr=sr_target)
    return y, sr_target

Uso el dataset filtrado porque no me alcanza la memoria para la CV

In [ ]:
archivos = df_filtrado['cycle_wav_file'].tolist()

mel_espectrograms = []

for archivo in archivos:
    path = os.path.join('./icbhi/ICBHI_ciclos/', archivo)
    y, sr = filter_and_resample(path, sr_target=16_000, cutoff_hz=4_000)
    S = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128)
    S_dB = librosa.power_to_db(S, ref=np.max)
    mel_espectrograms.append(S_dB)

Padding con silencio, uso la longitud máxima.

In [113]:
max_length = max([mel.shape[1] for mel in mel_espectrograms])

mel_espectrograms_padded = []
for mel in mel_espectrograms:
    if mel.shape[1] < max_length:
        pad_width = max_length - mel.shape[1]
        mel_padded = np.pad(mel, ((0, 0), (0, pad_width)), mode='constant')
    else:
        mel_padded = mel[:, :max_length]
    mel_espectrograms_padded.append(mel_padded)

In [114]:
X_mel = np.array(mel_espectrograms_padded)

In [115]:
target = df_filtrado['label'].values

In [116]:
X_train, X_test, y_train, y_test = train_test_split(X_mel, target, test_size=0.2, random_state=42, stratify=target)

### Modelos clásicos

Los modelos de sklearn no aceptan shapes como esas, hay que aplanar.

In [ ]:
n_samples, n_mels, n_frames = X_train.shape

X_train_flat = X_train.reshape(n_samples, n_mels * n_frames)
X_test_flat = X_test.reshape(X_test.shape[0], n_mels * n_frames)

Varios modelos como SVM mejoran su rendimiento con datos escalados.

In [ ]:
scaler = StandardScaler()
X_train_flat = scaler.fit_transform(X_train_flat)
X_test_flat = scaler.transform(X_test_flat)

#### Random Forest

In [141]:
rf = RandomForestClassifier(random_state=42, n_jobs=-1)

param_grid = {
    'max_depth': [5, 7, 10],
    'min_samples_split': [10, 30, 50],
    'min_samples_leaf': [10, 15, 20]
}

In [142]:
grid_search = GridSearchCV(rf, param_grid, cv=StratifiedKFold(n_splits=5), scoring='recall', n_jobs=1) # o auc-roc
grid_search.fit(X_train_flat, y_train)

,estimator,RandomForestC...ndom_state=42)
,param_grid,"{'max_depth': [5, 7, ...], 'min_samples_leaf': [10, 15, ...], 'min_samples_split': [10, 30, ...]}"
,scoring,'recall'
,n_jobs,1
,refit,True
,cv,StratifiedKFo...shuffle=False)
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,n_estimators,100


In [143]:
print("Best params:", grid_search.best_params_)
print("Best CV score:", grid_search.best_score_)

best_model = grid_search.best_estimator_

Best params: {'max_depth': 5, 'min_samples_leaf': 15, 'min_samples_split': 50}
Best CV score: 0.8225252200732506


In [149]:
pd.DataFrame(grid_search.cv_results_)[['mean_test_score', 'param_max_depth', 'param_min_samples_leaf', 'param_min_samples_split']].sort_values(by='mean_test_score', ascending=False)

,mean_test_score,param_max_depth,param_min_samples_leaf,param_min_samples_split
5,0.822525,5,15,50
0,0.821005,5,10,10
2,0.819983,5,10,50
8,0.818973,5,20,50
7,0.816436,5,20,30
6,0.816436,5,20,10
9,0.814917,7,10,10
13,0.814915,7,15,30
12,0.814915,7,15,10
1,0.814416,5,10,30


In [145]:
y_pred = best_model.predict(X_test_flat)
print(classification_report(y_test, y_pred))
print('AUC-ROC: ', roc_auc_score(y_test, y_pred))
print('Recall: ', recall_score(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.71      0.45      0.55       429
           1       0.64      0.84      0.72       493

    accuracy                           0.66       922
   macro avg       0.67      0.64      0.64       922
weighted avg       0.67      0.66      0.64       922

AUC-ROC:  0.6449713234703093
Recall:  0.8377281947261663
